In [ ]:
%load_ext autoreload
%autoreload 2

# Camera Localization

Given a 3D scene reconstructed by the feedforward pipeline, find where a new camera is positioned within it.

## §1 Setup

In [ ]:
import cv2
import numpy as np
import pyvista as pv
import torch
from pathlib import Path

from collab_splats.pointcloud.feedforward import VGGTXCreator
from collab_splats.pointcloud.feedforward.base import FeedforwardResult
from collab_splats.pointcloud.localization import CameraLocalizer, XFeatExtractor, plot_correspondences
from collab_splats.utils.frame_sampling import extract_video_frames, score_all_frames, save_frame_scores, load_frame_scores
from collab_splats.utils.visualization import create_camera_frustum_pyvista, pointcloud_to_polydata

pv.set_jupyter_backend("trame")

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
DATASET   = "birds_c0043"
METHOD    = "vggtx"    # "vggtx" | "mapanything"
VARIANT   = ""         # "ba" | "lc" | "" (empty = raw baseline)

from pathlib import Path
CACHE  = Path("../../.cache") / DATASET
IMAGES = CACHE / "images"
RECON  = CACHE / METHOD / VARIANT if VARIANT else CACHE / METHOD
RECON.mkdir(parents=True, exist_ok=True)

FRAME_DIR = CACHE / "images"

## §2 Reconstruct scene

In [ ]:
from collab_splats.pointcloud.feedforward.base import FeedforwardResult
result = FeedforwardResult.load_zarr(RECON / "reconstruction.zarr")
print(f"pts3d: {result.pts3d.shape}  frames: {result.extrinsics.shape[0]}")

## §3 Hold out query frame

In [ ]:
QUERY_IDX = 10

# Query frame — the camera we want to localize
query_bgr = cv2.imread(str(result.image_paths[QUERY_IDX]))
query_image = query_bgr[..., ::-1].copy()
query_K = result.intrinsics[QUERY_IDX]
ref_extrinsic = result.extrinsics[QUERY_IDX]  # feedforward reference for consistency check

# Scene without the query frame
keep = [i for i in range(len(result.image_paths)) if i != QUERY_IDX]

class _Trimmed:
    pts3d = result.pts3d          # shared — 3D points are not frame-specific
    extrinsics = result.extrinsics[keep]
    intrinsics = result.intrinsics[keep]
    image_paths = [result.image_paths[i] for i in keep]

result_trimmed = _Trimmed()
print(f"Scene: {len(result_trimmed.image_paths)} reference frames, query held out at index {QUERY_IDX}")

## §4 Localize

In [ ]:
localizer = CameraLocalizer.from_feedforward(result_trimmed, extractor=XFeatExtractor())
loc = localizer.localize(query_image, query_K)
print(f"correspondences: {loc.n_correspondences}  inliers: {loc.n_inliers}  pose: {'found' if loc.pose is not None else 'FAILED'}")
if loc.pose is None:
    raise RuntimeError("Localization failed — check n_correspondences above. Try a different QUERY_IDX.")

## §5 Correspondences

In [ ]:
plot_correspondences(loc, query_image, result_trimmed.image_paths)

## §6 3D view

In [ ]:
pl = pv.Plotter()
pl.add_mesh(pointcloud_to_polydata(result_trimmed.pts3d), point_size=2, render_points_as_spheres=True)
for ext in result_trimmed.extrinsics:
    pl.add_mesh(create_camera_frustum_pyvista(np.linalg.inv(ext), scale=0.05), color="grey", line_width=1)
pl.add_mesh(create_camera_frustum_pyvista(np.linalg.inv(loc.pose), scale=0.05), color="red", line_width=3)
pl.add_axes()
pl.show()

## §7 Error metrics

In [ ]:
R_est = loc.pose[:3, :3]
R_ref  = ref_extrinsic[:3, :3]
rot_err_deg = np.degrees(np.arccos(np.clip((np.trace(R_est @ R_ref.T) - 1) / 2, -1, 1)))
t_err_cm = np.linalg.norm(loc.pose[:3, 3] - ref_extrinsic[:3, 3]) * 100
print(f"vs feedforward reference — rotation: {rot_err_deg:.2f}°  |  translation: {t_err_cm:.1f} cm")
print("(consistency check vs feedforward model — not absolute ground truth)")